# Этап 4: структурированные признаки новости и контекста

Цель — проверить, улучшают ли четырёхчасовое abnormal direction дешёвые признаки типа события, числового surprise, сектора и рыночного режима до `decision_at`.

## План и замороженный критерий

Primary challenger — `reaction_core + structured_full`; context/news-only варианты являются ablations, TF-IDF — диагностический дешёвый benchmark. Evaluation: 480 событий, семь walk-forward folds, два месяца validation, embargo 72 часа, purge по `event_group_id`. Production-gate: hit rate ≥58%, нижняя cluster-CI hit rate >50%, минимум 5/7 положительных folds и нижняя cluster-CI Δ ROC-AUC против `reaction_core` >0.

In [ ]:
import json
import os
import sys
from pathlib import Path

import pandas as pd
from IPython.display import Image, display

cwd = Path.cwd()
experiment_dir = (
    cwd
    if (cwd / 'run_experiment.py').exists()
    else cwd / 'experiments' / 'direction_stage4'
)
repo_root = experiment_dir.parents[1]
sys.path.insert(0, str(repo_root))
from experiments.direction_stage4.build_structured_features import (  # noqa: E402
    _self_test_extractor,
    build_structured_features,
)
from experiments.direction_stage4.run_experiment import (  # noqa: E402
    assert_feature_contract,
    run_experiment,
)
from experiments.finbert_stage1.run_experiment import (  # noqa: E402
    load_dataset,
    sha256_file,
)
from experiments.paths import dataset_path_from_environment, stage_artifact_directory  # noqa: E402

dataset_path = dataset_path_from_environment()
artifact_dir = stage_artifact_directory('direction_stage4')
market_path = stage_artifact_directory('market_stage2') / 'market_features.csv'
structured_path = artifact_dir / 'structured_features.csv'
print(dataset_path)

In [ ]:
# Опциональная полная пересборка: RUN_STAGE4_BUILD=1 и RUN_STAGE4=1.
if os.environ.get('RUN_STAGE4_BUILD') == '1':
    _self_test_extractor()
    source_frame = load_dataset(dataset_path)
    market_frame = pd.read_csv(market_path)
    rebuilt = build_structured_features(source_frame, market_frame)
    rebuilt.to_csv(structured_path, index=False, float_format='%.10g')

if os.environ.get('RUN_STAGE4') == '1':
    result = run_experiment(
        dataset_path,
        artifact_dir,
        market_features_path=market_path,
        structured_features_path=structured_path,
    )
else:
    result = json.loads((artifact_dir / 'metrics.json').read_text())
feature_report = json.loads((artifact_dir / 'feature_build_report.json').read_text())
print(result['experiment'], result['created_at'])

## Автоматический контроль

In [ ]:
features = pd.read_csv(structured_path)
predictions = pd.read_csv(artifact_dir / 'direction_predictions.csv')
folds = pd.read_csv(artifact_dir / 'fold_metrics.csv')
assert_feature_contract()
_self_test_extractor()
assert len(features) == 1238 and features.id.nunique() == 1238
assert len(predictions) == 480 and predictions.id.nunique() == 480
assert predictions.fold.nunique() == 7
assert result['baseline_reproduction']['stage2_prediction_matches'] == 480
assert result['baseline_reproduction']['stage2_probability_max_abs_error'] < 1e-6
assert result['primary_deployment_gate']['passed'] is False
assert result['tfidf_deployment_gate']['passed'] is False
assert feature_report['structured_features_sha256'] == sha256_file(structured_path)
for filename, expected in result['artifact_hashes'].items():
    assert sha256_file(artifact_dir / filename) == expected
print(
    'QA passed: extractor self-tests, 1 238 feature rows, 480 OOF rows, '
    '7 folds, exact baseline classes, hashes and leakage contract.'
)

## Покрытие структурных признаков

In [ ]:
display(pd.Series(feature_report['coverage'], name='rate').to_frame().round(4))
examples = features.loc[
    (features.comparison_signal != 0) | (features.specific_event_signal != 0),
    [
        'event_subtype',
        'comparison_signal',
        'percent_change_signal',
        'specific_event_signal',
        'structured_signal',
        'model_text',
    ],
].head(8).copy()
examples['model_text'] = examples['model_text'].str.replace(r'\s+', ' ', regex=True).str[:180]
display(examples)

## Основные результаты

In [ ]:
rows = []
for name, metrics in result['metrics'].items():
    rows.append({
        'feature set': name,
        'ROC-AUC': metrics['roc_auc'],
        'hit rate': metrics['accuracy'],
        'balanced accuracy': metrics['balanced_accuracy'],
        'hit CI low': metrics['accuracy_cluster_bootstrap']['ci95_low'],
        'hit CI high': metrics['accuracy_cluster_bootstrap']['ci95_high'],
        'positive folds': metrics['positive_folds'],
    })
display(pd.DataFrame(rows).set_index('feature set').round(4))

In [ ]:
rows = []
for name, comparison in result['paired_comparisons_vs_reaction_core'].items():
    rows.append({
        'challenger': name,
        'Δ ROC-AUC': comparison['roc_auc_delta']['estimate'],
        'AUC Δ CI low': comparison['roc_auc_delta']['ci95_low'],
        'AUC Δ CI high': comparison['roc_auc_delta']['ci95_high'],
        'Δ hit rate': comparison['accuracy_delta']['estimate'],
        'hit Δ CI low': comparison['accuracy_delta']['ci95_low'],
        'hit Δ CI high': comparison['accuracy_delta']['ci95_high'],
    })
display(pd.DataFrame(rows).set_index('challenger').round(4))
print('Primary gate:', result['primary_deployment_gate'])

## Калибровка прямого сигнала и устойчивость по месяцам

In [ ]:
display(pd.DataFrame(result['feature_diagnostics']['structured_signal_calibration']).T)
fold_table = folds.pivot(index='fold', columns='feature_set', values='accuracy')
display(fold_table.round(4))
display(Image(filename=str(artifact_dir / 'model_comparison.png')))

## Config 6: одинаковые 157 событий

In [ ]:
config_rows = []
for name, metrics in result['config6_population']['models'].items():
    config_rows.append({
        'model': name,
        'hit rate': metrics['hit_rate'],
        'Δ vs config 6': metrics['delta_vs_config_6']['estimate'],
        'CI low': metrics['delta_vs_config_6']['ci95_low'],
        'CI high': metrics['delta_vs_config_6']['ci95_high'],
    })
print('config 6 hit rate:', result['config6_population']['config_6_hit_rate'])
display(pd.DataFrame(config_rows).set_index('model').round(4))

## Вывод

`structured_full` даёт только +0,0065 ROC-AUC с 95% CI [−0,0362; +0,0491], а hit rate снижается с 52,71% до 51,46%. Прямой structured signal слабо упорядочивает группы, но не позволяет надёжно определить знак отдельного события; TF-IDF ухудшает результат. Этап 4 не переносим в production. Последний рациональный дешёвый тест — небольшой nonlinear tabular model на ограниченном наборе надёжных признаков; при повторном провале direction исключается из продуктового обещания.